# Treino do modelo

## 0. Configs

### 0.1 Imports

In [7]:
import pandas as pd
import numpy as np
import warnings

# configurações para importar as funcões do módulo utils
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# funções utils
from src.utils_eda import months, days, faixa_etaria, sep_milhar

# configurações de exibição
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

### 0.2 Base de dados

In [ ]:
df = pd.read_parquet('../data/trusted/tabela_analitica.parquet', engine = 'pyarrow')
df = df.reset_index()
df['index'] = round(df['index'] / df['index'].max(), 3)

df.head()

,index,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,0.0,56,housemaid,married,basic.4y,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,0.0,57,services,married,high.school,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,0.0,37,services,married,high.school,Entre 31 e 40 anos,yes,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,0.0,40,admin.,married,basic.6y,Entre 31 e 40 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,0.0,56,services,married,high.school,Entre 50 e 60 anos,no,yes,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 1. Baseline

Variáveis em que foram observadas maior conversão:
- Faixa etária: mais de 60 anos e menos de 25 anos
- Profissão: estudante e aposentado
- Contato: celular
- Resultado da campanha anterior: success

In [ ]:
df_baseline = df[['faixa_etaria', 'job', 'contact', 'poutcome', 'y']].copy()

# baseline 1 (todas as características)
df_baseline['baseline1'] = np.where(
    (
        ((df_baseline['faixa_etaria'] == 'Mais de 60 anos') | (df_baseline['faixa_etaria'] == 'Até 25 anos')) &
        ((df_baseline['job'] == 'retired') | (df_baseline['job'] == 'student')) &
        (df_baseline['contact'] == 'cellular') &
        (df_baseline['poutcome'] == 'success')
    ),
    'yes',
    'no'
)

# baseline 2 (pelo menos 1 característica)
df_baseline['baseline2'] = np.where(
    (
        ((df_baseline['faixa_etaria'] == 'Mais de 60 anos') | (df_baseline['faixa_etaria'] == 'Até 25 anos')) |
        ((df_baseline['job'] == 'retired') | (df_baseline['job'] == 'student')) |
        (df_baseline['contact'] == 'cellular') |
        (df_baseline['poutcome'] == 'success')
    ),
    'yes',
    'no'
)

# baseline 3 (pelo menos 1 característica - removendo forma de contato)
df_baseline['baseline3'] = np.where(
    (
        ((df_baseline['faixa_etaria'] == 'Mais de 60 anos') | (df_baseline['faixa_etaria'] == 'Até 25 anos')) |
        ((df_baseline['job'] == 'retired') | (df_baseline['job'] == 'student')) |
        # (df_baseline['contact'] == 'cellular') |
        (df_baseline['poutcome'] == 'success')
    ),
    'yes',
    'no'
)

df_baseline


,faixa_etaria,job,contact,poutcome,y,baseline1,baseline2,baseline3,baseline4
0,Entre 50 e 60 anos,housemaid,telephone,nonexistent,no,no,no,no,no
1,Entre 50 e 60 anos,services,telephone,nonexistent,no,no,no,no,no
2,Entre 31 e 40 anos,services,telephone,nonexistent,no,no,no,no,no
3,Entre 31 e 40 anos,admin.,telephone,nonexistent,no,no,no,no,no
4,Entre 50 e 60 anos,services,telephone,nonexistent,no,no,no,no,no
...,...,...,...,...,...,...,...,...,...
41183,Mais de 60 anos,retired,cellular,nonexistent,yes,no,yes,yes,no
41184,Entre 41 e 50 anos,blue-collar,cellular,nonexistent,no,no,yes,no,no
41185,Entre 50 e 60 anos,retired,cellular,nonexistent,no,no,yes,yes,no
41186,Entre 41 e 50 anos,technician,cellular,nonexistent,yes,no,yes,no,no


In [68]:
print('Conversão geral')
df_baseline['y'].value_counts(normalize = True)

Conversão geral


y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

In [35]:
df_output = df_baseline[['y']].value_counts().reset_index().rename(columns = {'y':'output', 'count':'y'})

for i in range(1, 4):
    tmp = df_baseline[[f'baseline{i}']].value_counts().reset_index().rename(columns = {f'baseline{i}':'output', 'count':f'baseline{i}'})
    df_output = df_output.merge(tmp, on = 'output', how = 'outer')

df_output

,output,y,baseline1,baseline2,baseline3
0,no,36548,41007,13859,36152
1,yes,4640,181,27329,5036


In [64]:
def calcular_metricas_baseline(baseline, titulo):

    vp = df_baseline[(df_baseline['y'] == 'yes') & (df_baseline[baseline] == 'yes')].shape[0]
    fp = df_baseline[(df_baseline['y'] == 'no') & (df_baseline[baseline] == 'yes')].shape[0]
    vn = df_baseline[(df_baseline['y'] == 'no') & (df_baseline[baseline] == 'no')].shape[0]
    fn = df_baseline[(df_baseline['y'] == 'yes') & (df_baseline[baseline] == 'no')].shape[0]

    acuracia = (vp + vn) / (vp + fp + vn + fn)
    recall = (vp) / (vp + fn)
    precision = vp / (vp + fp)
    f1_score = 2 * (precision * recall) / (precision + recall)

    print(f"{baseline} | {titulo}\n{'-' * 100}\n")

    print(f"Total de observações:  {sep_milhar(vp + fp + vn + fn)}\n")
    print(f"Verdadeiros positivos: {sep_milhar(vp)}")
    print(f"Falsos positivos:      {sep_milhar(fp)}")
    print(f"Verdadeiros negativos: {sep_milhar(vn)}")
    print(f"Falsos negativos:      {sep_milhar(fn)}\n")
    print(f"Acurácia:              {round(acuracia, 4)}")
    print(f"Recall:                {round(recall, 4)} (vp / tudo que realmente é positivo)")
    print(f"Precision:             {round(precision, 4)} (vp / tudo que foi classificado positivo)")
    print(f"F1-score:              {round(f1_score, 4)} (média harmônica entre precision e recall)")

In [65]:
calcular_metricas_baseline(baseline = 'baseline1', titulo = 'Cliente tem todas as características')

baseline1 | Cliente tem todas as características
----------------------------------------------------------------------------------------------------

Total de observações:  41.188

Verdadeiros positivos: 136
Falsos positivos:      45
Verdadeiros negativos: 36.503
Falsos negativos:      4.504

Acurácia:              0.8896
Recall:                0.0293 (vp / tudo que realmente é positivo)
Precision:             0.7514 (vp / tudo que foi classificado positivo)
F1-score:              0.0564 (média harmônica entre precision e recall)


In [66]:
calcular_metricas_baseline(baseline = 'baseline2', titulo = 'Cliente tem pelo menos 1 das características')

baseline2 | Cliente tem pelo menos 1 das características
----------------------------------------------------------------------------------------------------

Total de observações:  41.188

Verdadeiros positivos: 4.012
Falsos positivos:      23.317
Verdadeiros negativos: 13.231
Falsos negativos:      628

Acurácia:              0.4186
Recall:                0.8647 (vp / tudo que realmente é positivo)
Precision:             0.1468 (vp / tudo que foi classificado positivo)
F1-score:              0.251 (média harmônica entre precision e recall)


In [67]:
calcular_metricas_baseline(baseline = 'baseline3' , titulo = 'Cliente tem pelo menos 1 das características (removendo forma de contato)')

baseline3 | Cliente tem pelo menos 1 das características (removendo forma de contato)
----------------------------------------------------------------------------------------------------

Total de observações:  41.188

Verdadeiros positivos: 1.623
Falsos positivos:      3.413
Verdadeiros negativos: 33.135
Falsos negativos:      3.017

Acurácia:              0.8439
Recall:                0.3498 (vp / tudo que realmente é positivo)
Precision:             0.3223 (vp / tudo que foi classificado positivo)
F1-score:              0.3355 (média harmônica entre precision e recall)


Conclusões do baseline:

- Como a conversão geral é de 11% (target muito desbalanceada), falar que ninguém vai converter já fornece uma acurácia de 89%, então não é uma boa métrica para ser acompanhada - olhar também precision e recall
- Apesar do baseline 1 ter uma acurácia melhor, o recall ficou muito baixo, o que fez o f1 score ficar muito baixo também
- O baseline 2 teve acurária de 42% (o que já é muito ruim), e a precision ficou ruim
- Para o baselne 3, precision e recall ficaram baixos, mas melhor que os demais baselines. Cabe ao modelo superar essas métricas
- O melhor baseline é o 3:
    - Job: retired ou student
    - Faixa etária: até 25 ou mais de 60
    - Resultado da campanha anterior: success

## 2. Treino do modelo

### 1.1 Separar em treino, validação e teste

In [14]:
# separação por fraçöFileExistsError
# train = df[df['index'] <= 0.7]
# valid = df[(df['index'] > 0.7) & (df['index'] <= 0.85)]
# test = df[df['index'] > 0.85]

# separação por ano
train = df[df['year'] == 2008]
valid = df[df['year'] == 2009]
test = df[df['year'] == 2010]

In [22]:
tamanho_inicial = df.shape[0]

print(f"Tamanho inicial:    {sep_milhar(tamanho_inicial)}\n")

print(f"- Treino (2008):    {sep_milhar(train.shape[0])} ({round(train.shape[0] * 100 / tamanho_inicial, 1)}%)")
print(f"- Validação (2009): {sep_milhar(valid.shape[0])} ({round(valid.shape[0] * 100 / tamanho_inicial, 1)}%)")
print(f"- Teste (2010):     {sep_milhar(test.shape[0])} ({round(test.shape[0] * 100 / tamanho_inicial, 1)}%)")

Tamanho inicial:    41.188

- Treino (2008):    27.690 (67.2%)
- Validação (2009): 11.440 (27.8%)
- Teste (2010):     2.058 (5.0%)


In [23]:
for tb in [train, valid, test]:
    print(tb['y'].value_counts(normalize = True))

y
no     0.951643
yes    0.048357
Name: proportion, dtype: float64
y
no     0.805245
yes    0.194755
Name: proportion, dtype: float64
y
yes    0.52138
no     0.47862
Name: proportion, dtype: float64


In [24]:
for tb in [train, valid, test]:
    print(tb['contact'].value_counts(normalize = True))

contact
cellular     0.504731
telephone    0.495269
Name: proportion, dtype: float64
contact
cellular     0.917657
telephone    0.082343
Name: proportion, dtype: float64
contact
cellular     0.811467
telephone    0.188533
Name: proportion, dtype: float64


In [20]:
train[['month']].value_counts().reset_index(name = 'treino_2008')\
    .merge(valid[['month']].value_counts().reset_index(name = 'valid_2009'), on = 'month', how = 'outer')\
    .merge(test[['month']].value_counts().reset_index(name = 'test_2010'), on = 'month', how = 'outer')

,month,treino_2008,valid_2009,test_2010
0,03. mar,NaN,282,264.0
1,04. apr,NaN,2458,174.0
2,05. may,7763.0,5794,212.0
3,06. jun,4374.0,715,229.0
4,07. jul,6685.0,178,311.0
5,08. aug,5175.0,770,233.0
6,09. sep,NaN,267,303.0
7,10. oct,67.0,447,204.0
8,11. nov,3616.0,357,128.0
9,12. dec,10.0,172,NaN
